# StudyB – Quality Assurance & Exclusion Script

## Quality Assurance & Exclusion Study B **PRIMARY**

In [ ]:
"""
StudyB - Quality Assurance & Exclusion Script

Reads the raw merged ratings file from Google Drive,
applies all pre-registered exclusion criteria, and saves
the cleaned dataset back to Google Drive.

Pre-registered exclusion criteria (Preregistration_StudyB.pdf, OSF-Prereg):
  1. IMC failed              -> imc_pass == 0
  2. Dialog time too short   -> min_time_ok_dialog == 0  (dialog_duration_sec < 10 s)
  3. Nutrient outlier        -> nutrient_outlier_flag == 1  (e.g. fat > 1000 g)
  4. Straightlining          -> ≥ 80% identical scale values across post + dialog-quality items
  5. Missing values          -> listwise deletion on primary outcome variables (MCAR assumption)

Usage

1. Mount your Google Drive in Colab:
       from google.colab import drive
       drive.mount('/content/drive')
2. Set the two path variables below (INPUT_PATH, OUTPUT_PATH).
3. Run the script. A cleaned CSV is written to OUTPUT_PATH.

Requirements: pandas, numpy  (both available by default in Colab)
"""

In [1]:
# Colab: Merge per-person rating CSVs into one file (NEW index)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
# Input and Output

INPUT_PATH  = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_merged.csv"
OUTPUT_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_cleaned.csv"

In [3]:
# configuration

# Scale items used for the straightlining check
SCALE_ITEMS = [
    "post_diet_suitability",
    "post_recipe_stars",
    "post_cook_intent",
    "post_save_intent",
    "dq_clarity",
    "dq_relevance",
    "dq_respect",
    "dq_logic",
    "dq_coherence",
    "manip_check",
]

# Straightlining threshold (pre-registration: ≥ 80 %)
STRAIGHTLINE_THRESHOLD = 0.80

# Primary outcome variables for listwise deletion
PRIMARY_OUTCOMES = [
    "post_diet_suitability",
    "post_recipe_stars",
    "post_cook_intent",
    "post_save_intent",
]

In [4]:
# helpers

import pandas as pd
import numpy as np


def compute_straightlining(row: pd.Series,
                            cols: list,
                            threshold: float = 0.80) -> bool:
    """
    Returns True if ≥ 'threshold' fraction of the available scale values
    in 'cols' are identical (i.e., the participant straight-lined).
    Requires at least 3 non-missing values to flag.
    """
    vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
    if len(vals) < 3:
        return False
    most_common_count = max(vals.count(v) for v in set(vals))
    return (most_common_count / len(vals)) >= threshold


def print_section(title: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

In [7]:
# main pipeline

def run_qa_pipeline(input_path: str, output_path: str) -> pd.DataFrame:

    # Load data
    print_section("1. Loading data")
    df = pd.read_csv(input_path)
    n_raw = len(df)
    print(f"  Rows loaded: {n_raw}")
    print(f"  Columns    : {df.shape[1]}")

    exclusion_log = []   # collects one dict per exclusion step

    # Criterion 1 - IMC
    print_section("2. Exclusion criterion 1: IMC failed (imc_pass == 0)")
    mask_imc = df["imc_pass"] == 0
    n_imc = mask_imc.sum()
    print(f"  Excluded: {n_imc}")
    if n_imc > 0:
        print(df.loc[mask_imc, ["rater_id", "condition", "recipe_title"]])
    exclusion_log.append({"criterion": "IMC failed", "n_excluded": int(n_imc)})
    df = df[~mask_imc].copy()

    # Criterion 2 - Dialog time
    print_section("3. Exclusion criterion 2: Dialog time too short (min_time_ok_dialog == 0)")
    mask_time = df["min_time_ok_dialog"] == 0
    n_time = mask_time.sum()
    print(f"  Excluded: {n_time}")
    if n_time > 0:
        print(df.loc[mask_time, ["rater_id", "dialog_duration_sec", "condition", "recipe_title"]])
    exclusion_log.append({"criterion": "Dialog time < 10 s", "n_excluded": int(n_time)})
    df = df[~mask_time].copy()

    # Criterion 3 - Nutrient outlier
    print_section("4. Exclusion criterion 3: Nutrient outlier flag (nutrient_outlier_flag == 1)")
    mask_outlier = df["nutrient_outlier_flag"] == 1
    n_outlier = mask_outlier.sum()
    print(f"  Excluded: {n_outlier}")
    if n_outlier > 0:
        print(df.loc[mask_outlier,
                     ["rater_id", "pre_est_fat_g", "post_est_fat_g",
                      "pre_est_carb_g", "post_est_carb_g",
                      "pre_est_kcal", "post_est_kcal", "condition"]])
    exclusion_log.append({"criterion": "Nutrient outlier", "n_excluded": int(n_outlier)})
    df = df[~mask_outlier].copy()

    # Criterion 4 - Straightlining
    print_section("5. Exclusion criterion 4: Straightlining (≥ 80 % identical scale values)")
    available_items = [c for c in SCALE_ITEMS if c in df.columns]
    df["_straightlining_flag"] = df.apply(
        compute_straightlining,
        axis=1,
        cols=available_items,
        threshold=STRAIGHTLINE_THRESHOLD,
    )
    mask_straight = df["_straightlining_flag"]
    n_straight = mask_straight.sum()
    print(f"  Scale items checked: {available_items}")
    print(f"  Excluded: {n_straight}")
    if n_straight > 0:
        for _, row in df[mask_straight].iterrows():
            vals = [row[c] for c in available_items if pd.notna(row.get(c))]
            print(f"    rater_id={row['rater_id']}, condition={row['condition']}, values={vals}")
    exclusion_log.append({"criterion": "Straightlining", "n_excluded": int(n_straight)})
    df = df[~mask_straight].drop(columns=["_straightlining_flag"]).copy()

    # Criterion 5 - Missing values (listwise deletion)
    print_section("6. Exclusion criterion 5: Missing values on primary outcomes (listwise)")
    missing_before = df[PRIMARY_OUTCOMES].isna().any(axis=1)
    n_missing = missing_before.sum()
    print(f"  Primary outcomes checked: {PRIMARY_OUTCOMES}")
    for col in PRIMARY_OUTCOMES:
        print(f"    {col}: {df[col].isna().sum()} missing")
    pct_missing = n_missing / len(df) * 100
    print(f"  Participants with ≥ 1 missing primary outcome: {n_missing} ({pct_missing:.1f} %)")
    if n_missing > 0:
        print(df.loc[missing_before, ["rater_id", "condition", "recipe_title"]])
    exclusion_log.append({"criterion": "Missing primary outcome (listwise)", "n_excluded": int(n_missing)})
    df = df[~missing_before].copy()

    # Summary
    print_section("7. Exclusion summary")
    total_excluded = n_raw - len(df)
    print(f"  Raw N              : {n_raw}")
    for entry in exclusion_log:
        print(f"  - {entry['criterion']:<38}: {entry['n_excluded']:>3} excluded")
    print(f"  {'─'*45}")
    print(f"  Total unique excluded (sequential): {total_excluded}")
    print(f"  Final cleaned N    : {len(df)}")

    # Condition balance after cleaning
    print("\n  Condition balance after cleaning:")
    print(df["condition"].value_counts().to_string())

    # Save
    print_section("8. Saving cleaned data")
    df.to_csv(output_path, index=False)
    print(f"  Cleaned CSV saved to:\n     {output_path}")
    print(f"  Columns: {df.shape[1]}  |  Rows: {df.shape[0]}")

    return df

In [8]:
# entry point ENTRY POINT

if __name__ == "__main__":
    df_clean = run_qa_pipeline(INPUT_PATH, OUTPUT_PATH)


  1. Loading data
  Rows loaded: 155
  Columns    : 47

  2. Exclusion criterion 1: IMC failed (imc_pass == 0)
  Excluded: 0

  3. Exclusion criterion 2: Dialog time too short (min_time_ok_dialog == 0)
  Excluded: 3
        rater_id  dialog_duration_sec condition  \
4    Ayushi Jain             7.508387       bad   
42        R27317             4.332140      good   
142       R94934             9.290343      good   

                              recipe_title  
4      Cuban-Style Pork and Sweet Potatoes  
42                        Carrot Casserole  
142  Orange Beef Kabobs with Grilled Fruit  

  4. Exclusion criterion 3: Nutrient outlier flag (nutrient_outlier_flag == 1)
  Excluded: 4
    rater_id  pre_est_fat_g  post_est_fat_g  pre_est_carb_g  post_est_carb_g  \
35    R22041           50.0            50.0            50.0             50.0   
66    R51031           38.0            38.0            20.0             20.0   
77    R60790           50.0            40.0           200.0     

## Quality Assurance & Exclusion Study B ***SENS***

In [9]:
def run_qa_pipeline_sens(input_path: str, output_path_sens: str) -> pd.DataFrame:
    """
    SENS dataset for Study B:
    - IMC failed -> still excluded (validity criterion, not a quality threshold)
    - Dialog time < 10s -> included
    - Nutrient outlier -> included
    - Straightlining -> included
    - Missing primary outcomes -> still excluded (listwise under MCAR)
    """
    df = pd.read_csv(input_path)
    n_raw = len(df)
    exclusion_log = []

    # Criterion 1: IMC - kept in SENS
    mask_imc = df["imc_pass"] == 0
    n_imc = mask_imc.sum()
    exclusion_log.append({"criterion": "IMC failed (kept in SENS)",
                          "n_excluded": int(n_imc)})
    df = df[~mask_imc].copy()

    # Criteria 2–4: Dialog time, nutrient outlier, straightlining -> NOT applied in SENS

    # Criterion 5: Missing primary outcomes - kept in SENS
    missing_before = df[PRIMARY_OUTCOMES].isna().any(axis=1)
    n_missing = missing_before.sum()
    exclusion_log.append({"criterion": "Missing primary outcome (listwise)",
                          "n_excluded": int(n_missing)})
    df = df[~missing_before].copy()

    print_section("SENS Exclusion summary")
    print(f"  Raw N              : {n_raw}")
    for entry in exclusion_log:
        print(f"  – {entry['criterion']:<45}: {entry['n_excluded']:>3} excluded")
    print(f"  Final SENS N       : {len(df)}")
    print("\n  Condition balance (SENS):")
    print(df["condition"].value_counts().to_string())

    df.to_csv(output_path_sens, index=False)
    print(f"\n SENS dataset saved to: {output_path_sens}")
    return df


# Path for SENS output
OUTPUT_PATH_SENS = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_cleaned_SENS.csv"

df_sens = run_qa_pipeline_sens(INPUT_PATH, OUTPUT_PATH_SENS)


  SENS Exclusion summary
  Raw N              : 155
  – IMC failed (kept in SENS)                    :   0 excluded
  – Missing primary outcome (listwise)           :   2 excluded
  Final SENS N       : 153

  Condition balance (SENS):
condition
good    78
bad     75

 SENS dataset saved to: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_cleaned_SENS.csv


## Merge LLM-Scores with Study B results for ***H6*** (PRIMARY & SENS)

In [10]:
# merge for PRIMARY
"""
Merge: studyB_all_ratings_merged.csv + llm_median_all_dialogs.csv

Joined on: dialog_id (left join — all StudyB participants are retained)

Duplicates in the LLM file (recipe_title, recipe_type, condition) are removed before
merge, as these columns are already present in the StudyB data.

Usage in Google Colab
---------------------
1. Mount Google Drive:
       from google.colab import drive
       drive.mount(“/content/drive”)
2. Adjust the three path variables below.
3. Run the script.
"""

STUDYB_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_cleaned.csv"
LLM_PATH    = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/llm_median_all_dialogs.csv"
OUTPUT_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_merged_with_llm.csv"

# merge

import pandas as pd

# read csv
studyB = pd.read_csv(STUDYB_PATH)
llm    = pd.read_csv(LLM_PATH)

print(f"StudyB:  {studyB.shape[0]} Zeilen, {studyB.shape[1]} Spalten")
print(f"LLM:     {llm.shape[0]} Zeilen, {llm.shape[1]} Spalten")

# Keep only the relevant LLM columns
# (Meta-columns that already exist in StudyB are left out)
llm_keep = [
    "dialog_id",
    "llm_median_clarity",
    "llm_median_relevance",
    "llm_median_truthfulness",
    "llm_median_logic_coherence",
    "llm_median_respect_appreciation",
    "llm_median_relational_appropriateness",
    "llm_median_feedback_depth",
    "llm_median_overall_quality",
    "n_runs",
    "median_valid",
]
llm_sub = llm[llm_keep].copy()

# Merge (left join on dialog_id)
merged = studyB.merge(llm_sub, on="dialog_id", how="left")

# Quality control
n_missing_llm = merged["llm_median_overall_quality"].isna().sum()
print(f"\nMerge completed:")
print(f"  Total rows : {len(merged)}")
print(f"  Total columns: {merged.shape[1]}")
print(f"  Missing LLM scores (dialog_ids with no match): {n_missing_llm}")

if n_missing_llm > 0:
    missing_ids = merged.loc[merged["llm_median_overall_quality"].isna(), "dialog_id"].unique()
    print(f"  Affected dialog_ids: {sorted(missing_ids)}")

StudyB:  139 Zeilen, 47 Spalten
LLM:     80 Zeilen, 19 Spalten

Merge completed:
  Total rows : 139
  Total columns: 57
  Missing LLM scores (dialog_ids with no match): 0


In [11]:
# Save
merged.to_csv(OUTPUT_PATH, index=False)
print(f"\n Saved: {OUTPUT_PATH}")


 Saved: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_merged_with_llm.csv


In [12]:
# Merge for SENS

STUDYB_SENS_PATH = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_all_ratings_cleaned_SENS.csv"
LLM_PATH         = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/llm_median_all_dialogs.csv"
OUTPUT_PATH_SENS = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_merged_with_llm_SENS.csv"

import pandas as pd

# read csv
studyB_sens = pd.read_csv(STUDYB_SENS_PATH)
llm         = pd.read_csv(LLM_PATH)

print(f"StudyB SENS: {studyB_sens.shape[0]} Zeilen, {studyB_sens.shape[1]} Spalten")
print(f"LLM:         {llm.shape[0]} Zeilen, {llm.shape[1]} Spalten")

# Relevant LLM Columns
llm_keep = [
    "dialog_id",
    "llm_median_clarity",
    "llm_median_relevance",
    "llm_median_truthfulness",
    "llm_median_logic_coherence",
    "llm_median_respect_appreciation",
    "llm_median_relational_appropriateness",
    "llm_median_feedback_depth",
    "llm_median_overall_quality",
    "n_runs",
    "median_valid",
]
llm_sub = llm[llm_keep].copy()

# Left Join on dialog_id
merged_sens = studyB_sens.merge(llm_sub, on="dialog_id", how="left")

# Quality Control
n_missing_llm = merged_sens["llm_median_overall_quality"].isna().sum()
print(f"\nMerge completed (SENS):")
print(f"  Total rows : {len(merged_sens)}")
print(f"  Total columns: {merged_sens.shape[1]}")
print(f"  Missing LLM scores (dialog_ids with no match): {n_missing_llm}")

if n_missing_llm > 0:
    missing_ids = merged_sens.loc[
        merged_sens["llm_median_overall_quality"].isna(), "dialog_id"
    ].unique()
    print(f"  Affected dialog_ids: {sorted(missing_ids)}")

# Save
merged_sens.to_csv(OUTPUT_PATH_SENS, index=False)
print(f"\nSaved: {OUTPUT_PATH_SENS}")

StudyB SENS: 153 Zeilen, 47 Spalten
LLM:         80 Zeilen, 19 Spalten

Merge completed (SENS):
  Total rows : 153
  Total columns: 57
  Missing LLM scores (dialog_ids with no match): 0

Saved: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/studyB_merged_with_llm_SENS.csv
